In [1]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Reshape, Bidirectional, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np


In [2]:
DATASET_PATH = r"C:\Users\abhil\Desktop\model_z\Data"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20   # MUST be same as baseline

train_dir = DATASET_PATH + r"\Train"
test_dir  = DATASET_PATH + r"\Test"

datagen = ImageDataGenerator(rescale=1./255)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

test_gen = datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)


Found 5435 images belonging to 2 classes.
Found 1359 images belonging to 2 classes.


In [3]:
print("Train samples:", train_gen.samples)
print("Test samples :", test_gen.samples)
print("Class indices:", train_gen.class_indices)


Train samples: 5435
Test samples : 1359
Class indices: {'Hemorrhagic': 0, 'NORMAL': 1}


In [4]:
base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze CNN
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

# 🔑 Convert features to sequence (timesteps = 1)
x = Reshape((1, x.shape[-1]))(x)

# 🔑 BiLSTM (DO NOT CHANGE PARAMETERS)
x = Bidirectional(LSTM(
    units=128,
    return_sequences=False,
    dropout=0.3,
    recurrent_dropout=0.3
))(x)

output = Dense(train_gen.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)


In [5]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 8,218,690 (31.35 MB)

 Trainable params: 1,181,186 (4.51 MB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [6]:
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=test_gen
)


c:\Users\abhil\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 544s 1s/step - accuracy: 0.6546 - loss: 0.6210 - val_accuracy: 0.7940 - val_loss: 0.4689
Epoch 2/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 418s 1s/step - accuracy: 0.7784 - loss: 0.4898 - val_accuracy: 0.8536 - val_loss: 0.3715
Epoch 3/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 571s 2s/step - accuracy: 0.8190 - loss: 0.4252 - val_accuracy: 0.8617 - val_loss: 0.3342
Epoch 4/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 612s 2s/step - accuracy: 0.8269 - loss: 0.3925 - val_accuracy: 0.9036 - val_loss: 0.2837
Epoch 5/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 553s 2s/step - accuracy: 0.8563 - loss: 0.3451 - val_accuracy: 0.8911 - val_loss: 0.2775
Epoch 6/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 809s 2s/step - accuracy: 0.8723 - loss: 0.3171 - val_accuracy: 0.9154 - val_loss: 0.2387
Epoch 7/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 861s 3s/step - accuracy: 0.8734 - loss: 0.3092 - val_accuracy: 0.9205 - val_loss: 0.2191
Epoch 8/20
340/340 ━━━━━━━━━━━━━━━━━━━━ 888s 3s/step - accuracy: 0.9000 - loss: 0.2661 - val_accu

In [7]:
y_true = test_gen.classes
y_pred_prob = model.predict(test_gen)
y_pred = np.argmax(y_pred_prob, axis=1)

accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="weighted")
recall    = recall_score(y_true, y_pred, average="weighted")
f1        = f1_score(y_true, y_pred, average="weighted")

print("DenseNet121 + BiLSTM Results:")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)


85/85 ━━━━━━━━━━━━━━━━━━━━ 75s 790ms/step
DenseNet121 + BiLSTM Results:
Accuracy : 0.9742457689477557
Precision: 0.9744177926959858
Recall   : 0.9742457689477557
F1-score : 0.9741710077212603
